In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
)
print(type(model))

/home/zl4063/Documents/github/ZO-LLM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [2]:
from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM

In [4]:
import inspect

In [5]:
inspect.getfile(Qwen2ForCausalLM)

'/home/zl4063/Documents/github/ZO-LLM/.venv/lib/python3.10/site-packages/transformers/models/qwen2/modeling_qwen2.py'

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
)

for idx, layer in enumerate(model.model.layers):
    mlp = layer.mlp

    # 注意：我们 hook 的是激活前乘 down_proj 之前的那一步，即 activation(gate_proj(x))
    def capture_gate_act_fn(module, input):
        x = input[0]
        act_output = module.act_fn(module.gate_proj(x))
        nonzero = (act_output != 0).float().sum()
        total = act_output.numel()
        ratio = nonzero / total
        print(f"[Layer {idx}] Non-zero activation ratio: {ratio.item():.4f}")
        return act_output * module.up_proj(x)  # 手动执行 SwiGLU

    # 替换掉 forward
    mlp.forward = lambda x, module=mlp: module.down_proj(
        capture_gate_act_fn(module, (x,))
    )


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
inputs = tokenizer("The quick brown fox jumps over the lazy dog", return_tensors="pt")
model.eval()
with torch.no_grad():
    model(**inputs)

[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27] Non-zero activation ratio: 1.0000
[Layer 27]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SmallGELUNet(nn.Module):
    def __init__(self, input_dim=16, hidden_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.GELU()  # 可以换成 SiLU/ReLU/GELU 进行对比
        self.fc2 = nn.Linear(hidden_dim, 8)

    def forward(self, x):
        x = self.fc1(x)
        self._last_activation = self.act(x)  # 保存 activation 输出以便外部分析
        x = self.fc2(self._last_activation)
        return x


def measure_activation_sparsity(tensor: torch.Tensor, eps=1e-4):
    near_zero = (tensor.abs() < eps).float().sum()
    total = tensor.numel()
    sparse_ratio = near_zero / total
    nonzero_ratio = 1.0 - sparse_ratio
    print(f"Approximate non-zero activation ratio: {nonzero_ratio:.4f}")
    return nonzero_ratio


model = SmallGELUNet()
model.eval()

# 随机输入
x = torch.randn(32, 16)  # batch size 32
with torch.no_grad():
    output = model(x)
    activation = model._last_activation
    measure_activation_sparsity(activation)

Approximate non-zero activation ratio: 1.0000
